# Experiment: Tiny-GPT2 Analytic Basis Sweep

This notebook summarizes the completed basis sweep for platonic initialization on the tiny demo setup.

Pipeline:
1. Use 8 Dyck pre-pretraining seeds (`sshleifer/tiny-gpt2`) to estimate shared subspace.
2. Fit analytic families to principal components.
3. Re-run init eval on WikiText-2 for each fitted family.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 120


def find_repo_root(start: Path | None = None) -> Path:
    cur = (start or Path.cwd()).resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root')

ROOT = find_repo_root()
ART = ROOT / 'artifacts' / 'demo_basis_sweep'
RUN_ROOT = ROOT / 'runs' / 'dyck_d10_5k_demo' / 'sshleifer_tiny-gpt2'

print('ROOT:', ROOT)
print('Artifacts dir exists:', ART.exists())
print('Run root exists:', RUN_ROOT.exists())


## Data + Run Sanity


In [ ]:
seed_dirs = sorted([p for p in RUN_ROOT.glob('seed_*') if p.is_dir()])
print('num_seed_runs:', len(seed_dirs))
print('seed_dirs:', [p.name for p in seed_dirs])

summary = json.loads((ART / 'weight_subspace_summary.json').read_text(encoding='utf-8'))
print('num_tensors_analyzed:', summary['num_tensors'])
print('total_params_analyzed:', summary['total_params_analyzed'])


## Load Basis Sweep Outputs


In [ ]:
BASIS = ['chebyshev', 'fourier', 'rbf', 'poly_exp']

fit_reports = {
    b: json.loads((ART / f'analytic_fit_report_{b}.json').read_text(encoding='utf-8'))
    for b in BASIS
}

evals = {
    b: json.loads((ART / f'init_eval_{b}.json').read_text(encoding='utf-8'))['results']
    for b in BASIS
}

rows = []
for b in BASIS:
    by_variant = {r['variant']: r for r in evals[b]}
    rows.append({
        'basis': b,
        'fit_mean_relative_error': fit_reports[b]['mean_relative_error'],
        'random_train_loss': by_variant['random']['train_loss'],
        'random_eval_loss': by_variant['random']['eval_loss'],
        'platonic_mean_train_loss': by_variant['platonic_mean']['train_loss'],
        'platonic_mean_eval_loss': by_variant['platonic_mean']['eval_loss'],
        'platonic_sampled_train_loss': by_variant['platonic_sampled']['train_loss'],
        'platonic_sampled_eval_loss': by_variant['platonic_sampled']['eval_loss'],
    })

summary_df = pd.DataFrame(rows).sort_values('basis').reset_index(drop=True)
summary_df['delta_mean_vs_random'] = summary_df['platonic_mean_eval_loss'] - summary_df['random_eval_loss']
summary_df['delta_sampled_vs_random'] = summary_df['platonic_sampled_eval_loss'] - summary_df['random_eval_loss']
summary_df


## Plot 1: Analytic Fit Error by Basis
Lower is better.


In [ ]:
plot_df = summary_df.sort_values('fit_mean_relative_error')
plt.figure(figsize=(7, 4))
sns.barplot(data=plot_df, x='basis', y='fit_mean_relative_error', palette='Blues_r')
plt.title('Mean Relative Reconstruction Error by Basis')
plt.xlabel('Basis Family')
plt.ylabel('Mean Relative Error')
plt.tight_layout()
plt.show()


## Plot 2: Eval Loss by Variant and Basis


In [ ]:
long_eval = summary_df.melt(
    id_vars=['basis'],
    value_vars=['random_eval_loss', 'platonic_mean_eval_loss', 'platonic_sampled_eval_loss'],
    var_name='variant',
    value_name='eval_loss',
)

variant_map = {
    'random_eval_loss': 'random',
    'platonic_mean_eval_loss': 'platonic_mean',
    'platonic_sampled_eval_loss': 'platonic_sampled',
}
long_eval['variant'] = long_eval['variant'].map(variant_map)

plt.figure(figsize=(9, 4.5))
sns.barplot(data=long_eval, x='basis', y='eval_loss', hue='variant', palette='Set2')
plt.title('Init Eval Loss Across Basis Families and Variants')
plt.xlabel('Basis Family')
plt.ylabel('Eval Loss')
plt.legend(title='Variant', loc='best')
plt.tight_layout()
plt.show()


## Plot 3: Delta vs Random (Eval Loss)
Negative is better than random.


In [ ]:
delta_df = summary_df.melt(
    id_vars=['basis'],
    value_vars=['delta_mean_vs_random', 'delta_sampled_vs_random'],
    var_name='delta_kind',
    value_name='delta_eval_loss',
)

delta_map = {
    'delta_mean_vs_random': 'platonic_mean - random',
    'delta_sampled_vs_random': 'platonic_sampled - random',
}
delta_df['delta_kind'] = delta_df['delta_kind'].map(delta_map)

plt.figure(figsize=(9, 4.5))
sns.barplot(data=delta_df, x='basis', y='delta_eval_loss', hue='delta_kind', palette='coolwarm')
plt.axhline(0.0, color='black', linewidth=1)
plt.title('Eval Loss Delta vs Random by Basis')
plt.xlabel('Basis Family')
plt.ylabel('Eval Loss Delta')
plt.legend(title='Comparison', loc='best')
plt.tight_layout()
plt.show()


## Ranked View (Sampled Variant)


In [ ]:
ranked = summary_df.sort_values('platonic_sampled_eval_loss')[
    ['basis', 'fit_mean_relative_error', 'platonic_sampled_eval_loss', 'delta_sampled_vs_random']
].reset_index(drop=True)
ranked.index = ranked.index + 1
ranked


## Interpretation

- `chebyshev` and `rbf` were strongest for `platonic_sampled` eval in this run.
- `platonic_mean` underperformed random for all bases here.
- Fit quality and downstream eval correlate somewhat, but not perfectly, so both should be tracked.
